In [ ]:
########## This code is used to pull the wind predictor variable data from RTMA for all profiles ##########

In [ ]:
# Import the Packages
import ee
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
import numpy as np
from collections import defaultdict
from tqdm.notebook import tqdm
ee.Initialize()

In [ ]:
############# Load in Base Data #############

In [ ]:
### Pull in SWORD Lines for GEE ###
SWORD_Nodes = ee.FeatureCollection('Insert_GEE_Asset_Path_Obstructed_Nodes_Base_Clean') # Update this file path

In [ ]:
###  Pull in the Dam File ###
Dams = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\Study_Dams.shp") # Update this file path

In [ ]:
############# Grab All Possible Profiles #############

In [ ]:
### All Potential Pull in Profiles of Interest ###
Potential_Profiles = pd.read_csv(r"F:\Insert_File_Path_Here\Usable_Profiles_List.csv") # Update this file path
Potential_Profiles

In [ ]:
## Creat a list of the Dams
DamsList = Potential_Profiles["Assgn_dam"].unique().tolist()
DamsList.sort()

In [ ]:
## Use Profiles of Interest 
ProfilesOfInterest = pd.merge(Potential_Profiles[["Month","Day", "Year", "Assgn_dam"]], Dams[["grod_id","zone"]], left_on="Assgn_dam", right_on= "grod_id", how="left")
ProfilesOfInterest = ProfilesOfInterest[["Month","Day", "Year", "Assgn_dam","zone"]]
ProfilesOfInterest

In [ ]:
##########################
### Pull RTMA Data (Wind) ###
##########################

In [ ]:
### Set Imagery Parameters ###
#Add Earth Engine dataset
RTMAProduct = ee.ImageCollection('NOAA/NWS/RTMA') #RTMA

# Set parameters for the images
date_start = ee.Date('2013-03-01')
date_end = ee.Date('2025-09-09')


## Filter Imagery Product by Date
RTMA_Data = RTMAProduct.filterDate(date_start, date_end)

## Some images don't have all the bands  -- we want to create a function to id and drop those
def missingBands(image):
    bands = image.bandNames().length()
    return image.set("NumberofBands", bands)

MapBandNos = RTMA_Data.map(missingBands)

# No Missing Bands
RTMA_Data_BandFilt = MapBandNos.filter(ee.Filter.eq('NumberofBands', ee.Number(13))).select(['TMP', 'WIND','SPFH','ACPC01'])

In [ ]:
# Fix Padded Zeros Issue & at UTC Time Target
ProfilesOfInterest['Month_Pad'] = ProfilesOfInterest['Month'].astype(str)
ProfilesOfInterest['Month_Pad'] = ProfilesOfInterest['Month_Pad'].str.pad(width=2, side='left', fillchar='0')
ProfilesOfInterest['Day_Pad'] = ProfilesOfInterest['Day'].astype(str)
ProfilesOfInterest['Day_Pad'] = ProfilesOfInterest['Day_Pad'].str.pad(width=2, side='left', fillchar='0')

# Pull Proper Image for Time Zone
ProfilesOfInterest['UTC_Time'] = np.nan
ProfilesOfInterest['UTC_Time'] = np.where((ProfilesOfInterest['zone'] == 'Eastern'), '15', ProfilesOfInterest['UTC_Time'])
ProfilesOfInterest['UTC_Time'] = np.where((ProfilesOfInterest['zone'] == 'Central'), '16', ProfilesOfInterest['UTC_Time'])
ProfilesOfInterest['UTC_Time'] = np.where((ProfilesOfInterest['zone'] == 'Mountain'), '17', ProfilesOfInterest['UTC_Time'])
ProfilesOfInterest['UTC_Time'] = np.where((ProfilesOfInterest['zone'] == 'Pacific'), '18', ProfilesOfInterest['UTC_Time'])

## Create ID Code
cols = ['Year', 'Month_Pad', 'Day_Pad', 'UTC_Time']
ProfilesOfInterest['IDString'] = ProfilesOfInterest[cols].apply(lambda row: ''.join(row.values.astype(str)), axis=1)
ProfilesOfInterest

In [ ]:
## Filter Images to Match Profiles
ImageList = ProfilesOfInterest["IDString"].unique().tolist()
RTMA_Filter = RTMA_Data_BandFilt.filter(ee.Filter.inList('system:index', ImageList))
RTMA_Filter.size().getInfo()

In [ ]:
## Define the Functions
# Create Functions for extracting image info
def addDate(image):
    image_date = ee.Date(image.date())
    image_date = ee.Number.parse(image_date.format('YYYYMMdd'))
    return image.addBands(ee.Image(image_date).rename('date').toInt())

def rasterExtraction(image):
    feature = image.sampleRegions(
        collection = SWORDSubset,
        scale = 2500
    )
    return feature

In [ ]:
### The following process can be little slow. So, it can be a good idea to split the dam list up and run it in sections. 
# This could even be down by creating several copies of the script and running them concurrently. 

In [ ]:
## Create Empty Dataframe to join to
RTMA_Data_all = pd.DataFrame()

for i in tqdm(DamsList[0:50]): ## Update the list here
    SWORDSubset = SWORD_Nodes.filter(ee.Filter.eq('Assgn_dam', i))
    ProfileSubset = ProfilesOfInterest[(ProfilesOfInterest["Assgn_dam"] == i)]
    ListofProfileDates = ProfileSubset["IDString"].unique().tolist()
    RTMA_Filter_Subset = RTMA_Filter.filter(ee.Filter.inList('system:index', ListofProfileDates))

    # Extract 
    DamRTMA = RTMA_Filter_Subset.select(['TMP', 'WIND']).map(addDate).map(rasterExtraction).flatten()

    if DamRTMA.size().getInfo() > 0: 
        # Get Column Info
        Column_df = list(SWORDSubset.limit(0).getInfo()['columns'])
        Column_df.extend(['TMP', 'WIND','date'])

        # Get nested list
        nested_list = DamRTMA.reduceColumns(ee.Reducer.toList(len(Column_df)),Column_df).values().get(0)
        RTMA_Loop_Data = nested_list.getInfo()

        # Create DF of data
        DamRTMA_df = pd.DataFrame(RTMA_Loop_Data, columns = [Column_df ])
        
        # Add to DF
        output = pd.concat([RTMA_Data_all, DamRTMA_df], ignore_index=True)
        
        # Save to Big DF 
        RTMA_Data_all = output

        # Print Progress
        print("Dam ", str(i), "completed.")
    
    else: 
        print("Dam ", str(i), "has no data and is skipped.")

print("Export RTMA Data -- Completed")

In [ ]:
## Save the DF to a CSV -- Note the naming if running in sections. I ended my runs with "_P#". Multiple CSVs are combined in the next set of scripts
RTMA_Data_all.to_csv(r"F:\Insert_File_Output_Path_Here\RTMA_Data_P1.csv") # Update this file path

## Preview
RTMA_Data_all